# 04 — Feature Engineering (refactored)

This is the condensed version of notebook 04 that uses the refactored `src/features.py` module. The actual feature computations are identical to the original notebook 04. This version just removes the inline function definitions and imports them from `src/`.

**Why the refactor:** the same feature functions are needed in notebook 08 (hierarchical) and notebook 09 (external validation). Keeping one source of truth in `src/features.py` avoids copy-paste drift.

**Equivalence guarantee:** the refactor was validated against the original inline implementation on 9306 feature values across all six modes. All values match exactly.

**Output is unchanged from the original notebook:** `data/processed/features_common_no_baro.csv` with the same 129 (or 141 if raw accelerometer is present) features per window.

## 1. Imports

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Add the project root to the import path so `src` is discoverable.
# This notebook lives in notebooks/, so the project root is one level up.
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import features as F
from src.data_loading import build_session_index

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
INTERIM_DIR   = PROJECT_ROOT / 'data' / 'interim'
REPORTS_DIR   = PROJECT_ROOT / 'reports' / 'features'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Window: {F.WINDOW_SAMPLES} samples ({F.WINDOW_SECONDS}s at {F.SAMPLING_RATE} Hz)')
print(f'Step:   {F.STEP_SAMPLES} samples ({F.STEP_SAMPLES / F.SAMPLING_RATE}s)')

Window: 250 samples (5.0s at 50 Hz)
Step:   125 samples (2.5s)


## 2. Inventory sessions

Discover every CSV under `data/interim/` and parse participant/mode/day from the filename. Anything new that lands in `data/interim/` (day-3 data, external participants) is picked up automatically.

In [2]:
session_index = build_session_index(INTERIM_DIR)
print(f'Found {len(session_index)} session files in {INTERIM_DIR}')

import pandas as pd
session_idx_df = pd.DataFrame(session_index)
print()
print('Sessions per participant / mode:')
print(session_idx_df.groupby(['participant', 'mode']).size()
      .unstack(fill_value=0).to_string())

Found 56 session files in C:\Projects\this project-transport-mode\data\interim

Sessions per participant / mode:
mode         bus  cycling  metro  stationary  tram  walking
participant                                                
athina         3        3      3           3     3        3
babis          3        3      3           3     3        3
xiomara        3        3      3           3     3        3


## 3. Featurise every session

For each session, slide a 5-second window with 50 percent overlap. Skip any window containing a sample marked `bad_sample`. The `compute_window_features` dispatcher handles all 129 features (or 141 if `acc_x`, `acc_y`, `acc_z` are present).

In [3]:
all_features = []
all_dropped  = []

for sess in session_index:
    meta = {
        'session_id':  sess['session_id'],
        'participant': sess['participant'],
        'mode':        sess['mode'],
        'mode_id':     F.MODE_TO_ID.get(sess['mode'], -1),
        'day':         sess['day'],
    }
    feat_rows, drop_rows = F.featurise_session(
        Path(sess['path']), meta, include_event_features=False
    )
    all_features.extend(feat_rows)
    all_dropped.extend(drop_rows)

features_df = pd.DataFrame(all_features)
dropped_df  = pd.DataFrame(all_dropped)

print(f'Total windows kept:    {len(features_df)}')
print(f'Total windows dropped: {len(dropped_df)}')
print()
print('Windows per participant / mode:')
print(features_df.groupby(['participant', 'mode']).size()
      .unstack(fill_value=0).to_string())

Total windows kept:    14513
Total windows dropped: 12

Windows per participant / mode:
mode         bus  cycling  metro  stationary  tram  walking
participant                                                
athina       881      727    906         703   810      689
babis        828      852    840         841   835      723
xiomara      614      916    803         967   810      768


## 4. Sanity checks

Same four checks as the original notebook: linacc std ordering across modes, walking dominant frequency in the cadence band, walking peak count near 10, and no NaN or inf values.

In [4]:
# Check 1: linacc_mag_std ordering
mode_means = features_df.groupby('mode')[['linacc_mag_std', 'gyro_mag_std']].median()
mode_means = mode_means.reindex(F.MODES)
print('Median linacc_mag_std and gyro_mag_std per mode:')
print(mode_means.to_string())
assert mode_means.loc['stationary', 'linacc_mag_std'] < mode_means.loc['walking', 'linacc_mag_std']
assert mode_means.loc['walking', 'linacc_mag_std'] > mode_means.loc['metro', 'linacc_mag_std']

Median linacc_mag_std and gyro_mag_std per mode:
            linacc_mag_std  gyro_mag_std
mode                                    
stationary        0.018587      0.005064
walking           1.598864      0.424353
cycling           0.885716      0.457741
metro             0.100371      0.014050
bus               0.274771      0.037767
tram              0.124954      0.016814


In [5]:
# Check 2: walking dominant frequency lands in the walking band
wf = features_df.loc[features_df['mode'] == 'walking', 'linacc_mag_dominant_freq']
print(f'Walking dominant frequency: median={wf.median():.2f} Hz, '
      f'IQR=({wf.quantile(0.25):.2f}, {wf.quantile(0.75):.2f}) Hz')
med = wf.median()
# Magnitude FFT can show the fundamental (~1.5-2.5 Hz) or its 2nd harmonic.
assert 1.0 <= med <= 5.0, f'Walking dominant freq median {med:.2f} Hz outside 1-5 Hz'

Walking dominant frequency: median=3.60 Hz, IQR=(1.60, 3.80) Hz


In [6]:
# Check 3: walking peak count near the step cadence rate
pc = features_df.groupby('mode')['linacc_mag_peak_count'].median().reindex(F.MODES)
print('Median linacc_mag_peak_count per mode:')
print(pc.to_string())
assert 6 <= pc['walking'] <= 15, f'Walking peak count {pc["walking"]} outside 6-15'
assert pc['walking'] > pc['metro'], 'Walking should have more peaks than metro'

Median linacc_mag_peak_count per mode:
mode
stationary    6.0
walking       8.0
cycling       5.0
metro         5.0
bus           5.0
tram          5.0


In [7]:
# Check 4: no NaN or inf
feature_cols = [c for c in features_df.columns
                if c not in ('session_id', 'participant', 'mode', 'mode_id', 'day',
                             'window_index', 'window_start_s', 'window_end_s', 'flagged_frac')]
n_features = len(feature_cols)
print(f'Total feature columns: {n_features}')

nan_counts = features_df[feature_cols].isna().sum()
inf_counts = np.isinf(features_df[feature_cols].values).sum(axis=0)

if nan_counts.sum() > 0:
    print('Features with NaN (replaced with 0):')
    print(nan_counts[nan_counts > 0].to_string())
    features_df[feature_cols] = features_df[feature_cols].fillna(0.0)
if inf_counts.sum() > 0:
    print(f'WARNING: {int(inf_counts.sum())} infinite values, replaced with 0')
    features_df[feature_cols] = features_df[feature_cols].replace([np.inf, -np.inf], 0.0)
print('No NaN or inf remaining:', not features_df[feature_cols].isna().any().any())

Total feature columns: 129
No NaN or inf remaining: True


## 5. Feature inventory and CSV outputs

Classify each feature by family and save the inventory plus the main feature table. The barometer ablation table is produced as before from sessions that include a `pressure` column.

In [8]:
def classify_feature(name: str) -> dict:
    fam = 'other'
    if name.startswith('event_'):
        fam = 'event_burstiness'
    elif any(s in name for s in ['_dominant_freq', '_dominant_amp', '_freq_weighted_avg',
                                  '_spec_entropy', '_power_band', '_band_ratio']):
        fam = 'frequency_domain'
    elif '_corr_' in name or name == 'linacc_gyro_mag_corr':
        fam = 'cross_axis_correlation'
    elif '_std_ratio_' in name:
        fam = 'axis_std_ratio'
    elif name.endswith('_sma'):
        fam = 'signal_magnitude_area'
    elif '_peak_count' in name:
        fam = 'peak_count'
    elif name.endswith('_zcr'):
        fam = 'zero_crossing_rate'
    elif name.endswith('_slope'):
        fam = 'slope'
    elif name.endswith('_skew') or name.endswith('_kurtosis') or name.endswith('_energy'):
        fam = 'shape_and_energy'
    elif any(name.endswith('_' + s) for s in ['mean', 'std', 'median', 'iqr',
                                                'rms', 'min', 'max', 'range']):
        fam = 'time_domain_basic'
    return {'feature': name, 'family': fam}


inventory = pd.DataFrame([classify_feature(c) for c in feature_cols])
print('Features per family:')
print(inventory.groupby('family').size().sort_values(ascending=False).to_string())
inventory.to_csv(REPORTS_DIR / 'feature_inventory.csv', index=False)

Features per family:
family
time_domain_basic         72
frequency_domain          20
shape_and_energy           9
slope                      8
cross_axis_correlation     7
axis_std_ratio             6
peak_count                 3
signal_magnitude_area      2
zero_crossing_rate         2


In [9]:
meta_cols = ['session_id', 'participant', 'mode', 'mode_id', 'day',
             'window_index', 'window_start_s', 'window_end_s', 'flagged_frac']
ordered = meta_cols + feature_cols
features_df = features_df[ordered]

out_path = PROCESSED_DIR / 'features_common_no_baro.csv'
features_df.to_csv(out_path, index=False)
print(f'Saved {len(features_df)} rows x {len(features_df.columns)} cols to {out_path}')

if len(dropped_df) > 0:
    dropped_df.to_csv(REPORTS_DIR / 'windows_dropped.csv', index=False)
    print(f'Saved {len(dropped_df)} dropped windows to windows_dropped.csv')

Saved 14513 rows x 138 cols to C:\Projects\this project-transport-mode\data\processed\features_common_no_baro.csv
Saved 12 dropped windows to windows_dropped.csv


## 6. Optional: produce the extended feature table (with event features)

Run this cell only if you plan to use notebook 08 (hierarchical model). It adds 12 burstiness features per window for the stationary vs vehicle distinction. The output is a separate file so the original `features_common_no_baro.csv` is not overwritten.

Skip this cell otherwise. Notebook 05's results stay unchanged either way.

In [10]:
all_features_ext = []
for sess in session_index:
    meta = {
        'session_id':  sess['session_id'],
        'participant': sess['participant'],
        'mode':        sess['mode'],
        'mode_id':     F.MODE_TO_ID.get(sess['mode'], -1),
        'day':         sess['day'],
    }
    feat_rows, _ = F.featurise_session(
        Path(sess['path']), meta, include_event_features=True
    )
    all_features_ext.extend(feat_rows)

features_ext_df = pd.DataFrame(all_features_ext)
ext_feature_cols = [c for c in features_ext_df.columns if c not in meta_cols]
features_ext_df[ext_feature_cols] = (features_ext_df[ext_feature_cols]
                                       .fillna(0.0)
                                       .replace([np.inf, -np.inf], 0.0))
features_ext_df = features_ext_df[meta_cols + ext_feature_cols]

ext_path = PROCESSED_DIR / 'features_common_no_baro_extended.csv'
features_ext_df.to_csv(ext_path, index=False)
print(f'Saved {len(features_ext_df)} rows x {len(features_ext_df.columns)} cols to {ext_path}')
print(f'Total features: {len(ext_feature_cols)} ({len(ext_feature_cols) - len(feature_cols)} event features added)')

Saved 14513 rows x 150 cols to C:\Projects\this project-transport-mode\data\processed\features_common_no_baro_extended.csv
Total features: 141 (12 event features added)


## 7. Summary

Two files produced:
1. `data/processed/features_common_no_baro.csv` — the standard feature table (unchanged from the original notebook 04). Used by notebook 05.
2. `data/processed/features_common_no_baro_extended.csv` — the extended table with 12 additional burstiness features. Used by notebook 08 for the hierarchical model.

Both files are produced from the same windows so they have identical row counts and metadata. The extended table is a strict superset of the standard table.